# 31 — Multi-Agent Debate

Two LLM agents argue opposing sides of a question, exchange critiques for N rounds, then a judge selects the stronger position.

**What you'll learn**
- Two independent agents with different system prompts → genuinely different perspectives
- Critique exchange: how one agent's position becomes another agent's input
- Round-based stopping condition in `DebateState`
- Judge pattern: structured verdict with a parseable `WINNER:` tag
- Why debate improves factuality vs single-agent generation (Du et al. 2023)

**Companion examples:** 5-react-agent-lg (two-agent critic loop), 18-self-reflecting-agent (self-critique)

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q langchain langchain-openai python-dotenv langgraph

In [ ]:
import os

from dotenv import load_dotenv

if not IN_COLAB:
    load_dotenv()
if IN_COLAB and not os.environ.get("OPENAI_API_KEY"):
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running"

In [ ]:
# ── 3. Why debate works ────────────────────────────────────────────────────────
# Each agent has a distinct system prompt locking it into a role.
# This stops them converging to the same answer immediately.
#
# The critique exchange forces each agent to:
#   1. Read the opponent's strongest points
#   2. Acknowledge them (steelmanning)
#   3. Defend their own position more precisely
#
# Du et al. 2023 showed this improves factuality on math and reasoning tasks.

print("Flow per round:")
print("  solve_a (position) → solve_b (position) → debate (mutual critique)")
print("  → if round < MAX_ROUNDS: back to solve_a (refine using critique)")
print("  → if round >= MAX_ROUNDS: judge (pick winner)")

In [ ]:
# ── 4. Build the debate graph ─────────────────────────────────────────────────
from typing import TypedDict

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, StateGraph

MAX_ROUNDS = 2
TOPIC = "Is specialization or generalization better for a software engineer's career?"

SOLVER_A_SYSTEM = (
    "You are a career expert arguing FOR SPECIALIZATION in software engineering. "
    "Make compelling, evidence-based arguments in 3-4 sentences. "
    "If given a critique, acknowledge its strongest point then reinforce your position."
)
SOLVER_B_SYSTEM = (
    "You are a career expert arguing FOR GENERALIZATION (T-shaped or full-stack engineer). "
    "Make compelling, evidence-based arguments in 3-4 sentences. "
    "If given a critique, acknowledge its strongest point then reinforce your position."
)
JUDGE_SYSTEM = (
    "You are an impartial judge. Read both final positions, select the stronger argument, "
    "explain your reasoning in 2-3 sentences, then end with exactly: "
    "WINNER: [SPECIALIZATION or GENERALIZATION]"
)


class DebateState(TypedDict):
    topic: str
    position_a: str
    position_b: str
    critique_a: str  # B's critique of A
    critique_b: str  # A's critique of B
    round: int
    verdict: str


llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)


def solve_a(state: DebateState) -> dict:
    if state.get("critique_a"):
        content = (
            f"Topic: {state['topic']}\n\nYour current position: {state['position_a']}\n\n"
            f"Opponent's critique: {state['critique_a']}\n\nRefine your argument:"
        )
    else:
        content = f"Topic: {state['topic']}\n\nPresent your opening argument FOR SPECIALIZATION:"
    result = llm.invoke([SystemMessage(SOLVER_A_SYSTEM), HumanMessage(content)])
    print(f"  [A] {result.content[:80]}...")
    return {"position_a": result.content}


def solve_b(state: DebateState) -> dict:
    if state.get("critique_b"):
        content = (
            f"Topic: {state['topic']}\n\nYour current position: {state['position_b']}\n\n"
            f"Opponent's critique: {state['critique_b']}\n\nRefine your argument:"
        )
    else:
        content = f"Topic: {state['topic']}\n\nPresent your opening argument FOR GENERALIZATION:"
    result = llm.invoke([SystemMessage(SOLVER_B_SYSTEM), HumanMessage(content)])
    print(f"  [B] {result.content[:80]}...")
    return {"position_b": result.content}


def debate(state: DebateState) -> dict:
    crit_a = llm.invoke(
        [
            SystemMessage(SOLVER_A_SYSTEM),
            HumanMessage(f"Opponent argues: {state['position_b']}\n\nWrite a 2-sentence critique:"),
        ]
    )
    crit_b = llm.invoke(
        [
            SystemMessage(SOLVER_B_SYSTEM),
            HumanMessage(f"Opponent argues: {state['position_a']}\n\nWrite a 2-sentence critique:"),
        ]
    )
    rnd = state.get("round", 0) + 1
    print(f"  [debate round {rnd}] critiques generated")
    return {"critique_a": crit_a.content, "critique_b": crit_b.content, "round": rnd}


def should_continue(state: DebateState) -> str:
    return "solve_a" if state.get("round", 0) < MAX_ROUNDS else "judge"


def judge(state: DebateState) -> dict:
    content = (
        f"Topic: {state['topic']}\n\n"
        f"POSITION A (Specialization):\n{state['position_a']}\n\n"
        f"POSITION B (Generalization):\n{state['position_b']}"
    )
    result = llm.invoke([SystemMessage(JUDGE_SYSTEM), HumanMessage(content)])
    return {"verdict": result.content}


graph = StateGraph(DebateState)
graph.add_node("solve_a", solve_a)
graph.add_node("solve_b", solve_b)
graph.add_node("debate", debate)
graph.add_node("judge", judge)
graph.set_entry_point("solve_a")
graph.add_edge("solve_a", "solve_b")
graph.add_edge("solve_b", "debate")
graph.add_conditional_edges("debate", should_continue, {"solve_a": "solve_a", "judge": "judge"})
graph.add_edge("judge", END)
app = graph.compile()
print("Debate graph compiled — nodes: solve_a, solve_b, debate, judge")

In [ ]:
# ── 5. Run the debate ─────────────────────────────────────────────────────────
print(f"TOPIC: {TOPIC}\n")
result = app.invoke(
    {
        "topic": TOPIC,
        "position_a": "",
        "position_b": "",
        "critique_a": "",
        "critique_b": "",
        "round": 0,
        "verdict": "",
    }
)
print("\n=== FINAL POSITION A (Specialization) ===")
print(result["position_a"])
print("\n=== FINAL POSITION B (Generalization) ===")
print(result["position_b"])
print("\n=== JUDGE'S VERDICT ===")
print(result["verdict"])

## Exercises

**Exercise 1 — Change the topic:** Replace `TOPIC` with `"Should AI replace human code review?"`. How does the debate quality differ on a more contentious question?

**Exercise 2 — More rounds:** Change `MAX_ROUNDS = 3`. Do positions sharpen or repeat? Where does debate add diminishing returns?

**Exercise 3 — Three-way debate:** Add `SOLVER_C_SYSTEM` arguing a middle ground. Add `position_c` to `DebateState`. How does the judge handle three positions?

**Exercise 4 — Parse the winner:** Extract the `WINNER:` tag from `result['verdict']` with a regex. Run 5 different topics and track which side wins more often.

## What's next

- **18-self-reflecting-agent** — single agent critiques its own output (no opponent)
- **5-react-agent-lg** — two-agent critic loop: summarizer and reviewer
- **43-supervisor-worker** — supervisor dynamically routes to a pool of specialists